<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller: La Alarma de Humo y las Notas Justas — Métricas y Ajuste 🔔🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo · Módulo 03
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/Para%20Dummies/03_Metricas_y_Ajuste_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎈 ¿Qué vamos a hacer en este taller?

Una empresa instala **alarmas de incendio** en bodegas. Casi siempre no hay fuego (¡qué bueno!), pero justo por eso medir bien la alarma es difícil. Vas a:

1. 🔔 **Evaluar la alarma con la matriz de los cuatro resultados** y descubrir por qué «94 % de aciertos» puede ser una trampa.
2. 🔄 **Probar varias recetas sin hacer trampa:** validación cruzada (simulacros rotativos) y `GridSearchCV` (el cocinero que prueba todas las recetas) para elegir cuántos vecinos mira un modelo.
3. ✍️ **Contar con tus palabras** qué nota importa más.

> 🧸 **Cómo funciona:** cada reto trae un cuadro de código con huecos `...` y pistas 💡. Reemplaza los `...` por lo que corresponda. Debajo hay una **autoverificación** que te orienta con cariño. No traen soluciones: ¡inténtalo tú!

### 📋 Instrucciones

- Completa solo los huecos `...`; no cambies las celdas de preparación ni de autoverificación.
- Al final usa *Restart & Run All*: todo debe correr sin errores y sin `...` pendientes.
- Todo funciona sin internet (datos inventados con semilla fija). Repasa si quieres: [Métricas de Clasificación para Dummies](../../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/Para%20Dummies/00_Metricas_Clasificacion_Dummies.ipynb) y [Validación Cruzada y GridSearch para Dummies](../../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/Para%20Dummies/02_Validacion_Cruzada_GridSearch_Dummies.ipynb).

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos inventados
# ============================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score

# 1 200 lecturas de sensores de bodega; fuego = 1 (poco frecuente), sin fuego = 0
rng = np.random.default_rng(12)
n = 1200
humo = rng.normal(0, 1, n)
calor = rng.normal(0, 1, n)
fuego = rng.binomial(1, 1 / (1 + np.exp(-(-5.0 + 2.2 * humo + 1.7 * calor))))
bodegas = pd.DataFrame({"humo": humo, "calor": calor, "fuego": fuego})
X_train, X_test, y_train, y_test = train_test_split(bodegas[["humo", "calor"]], bodegas["fuego"],
                                                    test_size=0.25, stratify=bodegas["fuego"], random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # los 5 «simulacros rotativos»

print("Lecturas:", bodegas.shape, "| Proporción con fuego:", round(bodegas["fuego"].mean(), 3))
display(bodegas.head(3))

---
## Reto 1: Los cuatro resultados de la alarma y la trampa del 94 % 🔔

Una alarma puede acertar de cuatro maneras: **TP** (hubo fuego y sonó), **TN** (no hubo y no sonó), **FP** (sonó sin fuego: falsa alarma) y **FN** (hubo fuego y *no* sonó: ¡el peor!). Con ellos:

- **exactitud** = (TP + TN) / total · **cobertura** (*recall*) = TP / (TP + FN) · **precisión** = TP / (TP + FP).

**Pasos**
1. Ajusta `modelo = LogisticRegression()` con lo de estudiar y obtén `y_pred` para el examen (`X_test`).
2. Con `confusion_matrix(y_test, y_pred).ravel()` obtén `tn, fp, fn, tp` (en ese orden).
3. Calcula `exactitud`, `cobertura` y `precision` con las fórmulas.
4. `acc_siempre_no`: exactitud de una «alarma» que **nunca suena** (predice siempre 0).

**Puntaje sugerido:** 35 puntos.

In [ ]:
# Casillas que debes llenar (empiezan vacías; no las borres)
modelo = y_pred = None
tn = fp = fn = tp = None
exactitud = cobertura = precision = acc_siempre_no = None

# 💡 1) Ajustar y predecir en el examen
modelo = LogisticRegression()
modelo.fit(..., ...)
y_pred = modelo.predict(...)

# 💡 2) confusion_matrix devuelve [[TN, FP], [FN, TP]]; .ravel() lo aplana en ese orden
tn, fp, fn, tp = confusion_matrix(..., ...).ravel()

# 💡 3) Fórmulas
exactitud = (tp + tn) / (tp + tn + fp + fn)
cobertura = tp / (tp + ...)
precision = tp / (tp + ...)

# 💡 4) La alarma que nunca suena acierta siempre que no hay fuego: ¿qué fracción de y_test son ceros?
acc_siempre_no = np.mean(y_test == ...)

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert isinstance(modelo, LogisticRegression) and hasattr(modelo, "coef_"), "Pista 🧸: modelo = LogisticRegression() y luego modelo.fit(X_train, y_train)."
assert y_pred is not None and len(y_pred) == len(y_test), "y_pred = modelo.predict(X_test): una predicción por cada lectura del examen."
_tn, _fp, _fn, _tp = confusion_matrix(y_test, y_pred).ravel()
assert (tn, fp, fn, tp) == (_tn, _fp, _fn, _tp), "Pista 🧸: confusion_matrix(y_test, y_pred).ravel() devuelve los cuatro números en el orden tn, fp, fn, tp."
assert abs(exactitud - (_tp + _tn) / len(y_test)) < 1e-9, "exactitud = (tp + tn) / total de lecturas."
assert abs(cobertura - _tp / (_tp + _fn)) < 1e-9, "cobertura = tp / (tp + fn): del fuego que hubo, ¿cuánto detectó la alarma?"
assert abs(precision - _tp / (_tp + _fp)) < 1e-9, "precision = tp / (tp + fp): cuando sonó, ¿cuántas veces había fuego de verdad?"
assert abs(acc_siempre_no - (1 - y_test.mean())) < 1e-9, "La alarma que nunca suena acierta en todos los casos de y_test que son 0: np.mean(y_test == 0)."
assert 0 <= exactitud - acc_siempre_no < 0.05 and cobertura < 0.6, "Así debe verse la trampa: la exactitud casi no supera a la alarma que nunca suena, y la cobertura es baja."
print(f"✅ ¡Reto 1 superado! Exactitud {exactitud:.2f} (¡la alarma que nunca suena logra {acc_siempre_no:.2f}!), pero solo detecta el {cobertura:.0%} de los incendios.")

---
## Reto 2: Simulacros rotativos y el cocinero que prueba todas las recetas 🔄

Para elegir cuántos vecinos mira el modelo **k-NN** («pregúntale a los *k* datos más parecidos»), no se vale usar el examen final: sería ver las respuestas antes. Se usan **simulacros rotativos** (validación cruzada) dentro de lo de estudiar, y se deja el examen para el final.

**Pasos**
1. `pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())` (¡el escalador va **dentro**, para no filtrar información!).
2. `notas = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")` son las 5 notas de los simulacros; `nota_media` es su promedio.
3. `grid = GridSearchCV(pipe, {...}, cv=cv, scoring="roc_auc")` probando `n_neighbors` en `[1, 3, 5, 9, 15, 25, 51]` (en un pipeline el parámetro se llama `kneighborsclassifier__n_neighbors`). Ajústalo solo con lo de estudiar.
4. `mejor_k`: el valor ganador (`grid.best_params_`); `nota_examen`: el AUC del `grid` en el examen, **una sola vez**.

**Puntaje sugerido:** 40 puntos.

In [ ]:
# Casillas que debes llenar
pipe = notas = grid = None
nota_media = mejor_k = nota_examen = None

# 💡 1) El escalador y el modelo, juntos
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

# 💡 2) 5 simulacros rotativos SOLO con lo de estudiar
notas = cross_val_score(pipe, ..., ..., cv=cv, scoring="roc_auc")
nota_media = notas.mean()

# 💡 3) Probar todas las recetas: la clave del diccionario es 'kneighborsclassifier__n_neighbors'
grid = GridSearchCV(pipe, {"kneighborsclassifier__n_neighbors": [...]}, cv=cv, scoring="roc_auc")
grid.fit(..., ...)

# 💡 4) La receta ganadora y la nota del examen final
mejor_k = grid.best_params_["kneighborsclassifier__n_neighbors"]
nota_examen = roc_auc_score(y_test, grid.predict_proba(...)[:, 1])

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert pipe is not None and [type(p).__name__ for _, p in pipe.steps] == ["StandardScaler", "KNeighborsClassifier"], "Pista 🧸: make_pipeline(StandardScaler(), KNeighborsClassifier()): primero el escalador, luego el modelo."
assert notas is not None and len(notas) == 5, "notas debe tener 5 valores (uno por simulacro): cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')."
assert np.allclose(notas, cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")), "Las notas deben calcularse con X_train, y_train (lo de estudiar), el cv dado y scoring='roc_auc'."
assert nota_media is not None and abs(nota_media - np.mean(notas)) < 1e-12, "nota_media es el promedio de notas: notas.mean()."
assert isinstance(grid, GridSearchCV) and hasattr(grid, "best_params_"), "Pista 🧸: grid = GridSearchCV(pipe, {...}, cv=cv, scoring='roc_auc') y luego grid.fit(X_train, y_train)."
assert len(grid.cv_results_["params"]) == 7, "Debes probar 7 valores de k: [1, 3, 5, 9, 15, 25, 51]."
assert mejor_k == grid.best_params_["kneighborsclassifier__n_neighbors"], "mejor_k sale de grid.best_params_['kneighborsclassifier__n_neighbors']."
assert mejor_k > 1, "k = 1 memoriza el ruido y no debería ganar: revisa que ajustaste con X_train, y_train y puntuaste con roc_auc."
assert abs(nota_examen - roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1])) < 1e-9 and nota_examen > 0.8, "nota_examen es el AUC de grid en el examen (X_test, y_test) con las probabilidades de la clase 1."
assert abs(grid.best_score_ - nota_examen) < 0.15, "La nota de los simulacros y la del examen deberían parecerse (diferencia < 0.15)."
print(f"✅ ¡Reto 2 superado! Simulacros: {nota_media:.2f} ± {notas.std():.2f} | mejor k = {mejor_k} | examen final: {nota_examen:.2f}")

---
## Reto 3: ¿Qué nota importa más? ✍️

**Puntaje sugerido:** 25 puntos (sin código). Escribe **3 a 5 líneas** en la celda de abajo respondiendo:

- Para una alarma de incendio, ¿qué te importa más: **cobertura** (no dejar pasar incendios) o **precisión** (que no suene sin motivo)? ¿Por qué?
- ¿Por qué la «alarma que nunca suena» saca una nota tan alta en exactitud y aun así es inútil?
- ¿Por qué el examen final se usa **una sola vez** y no para elegir *k*?

---
#### ✍️ Tu respuesta

_Escribe aquí tus 3-5 líneas. Reemplaza este texto por tu reflexión._

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: cuatro resultados, métricas y la trampa de la exactitud | 35 |
| Reto 2: validación cruzada, `GridSearchCV` dentro de un `Pipeline` y examen final | 40 |
| Reto 3: reflexión con tus cifras | 25 |
| **Total** | **100** |

**Checklist:** ✅ ejecuté *Run All* sin errores · ✅ no quedan `...` · ✅ las dos autoverificaciones imprimen ✅ · ✅ escribí mi reflexión.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>